# DQN — Atari DRL Playtester

**Projeto:** Avaliação Automática de Dificuldade em Jogos Atari 2600 com Deep RL
**Disciplina:** Redes Neurais Artificiais — PPGCC/UNESP

Treina o agente **DQN** em 4 jogos Atari de dificuldade canônica crescente
× 3 seeds (42, 123, 2024), sob orçamento computacional fixo de
**500 000 timesteps** por configuração.

Jogos (ordenados por dificuldade reconhecida na literatura de RL):

| # | Jogo | Dificuldade | Referência |
|---|------|-------------|-----------|
| 1 | Pong | Fácil — resolvido em ~100k ts por DQN | Mnih et al. 2015 |
| 2 | Breakout | Médio — convergência em ~500k–1M ts | Mnih et al. 2015 |
| 3 | Ms.Pac-Man | Difícil — dinâmico, exploração ampla | Bellemare et al. 2016 |
| 4 | Montezuma's Revenge | Extremo — *sparse reward*, *hard exploration* | Badia et al. 2020 (Agent57) |

Stack moderna: `gymnasium ≥ 1.0` + `ale-py ≥ 0.10` (ROMs já incluídas no pacote)
+ `stable-baselines3 ≥ 2.4`. Python 3.10+, NumPy 2.x, **zero workarounds**.

## 1. Instalação

In [ ]:
# --- 1. Instalação ---
# Stack moderna real: gymnasium + ale-py + stable-baselines3.
# ale-py ≥ 0.10 (2024+) traz as ROMs do Atari empacotadas — sem AutoROM.
#
# Idempotente: detecta o que já está instalado e pula. Se algo for
# instalado/atualizado AGORA, reinicia o kernel automaticamente para
# evitar conflito numpy-em-memória vs numpy-em-disco (bug clássico do Colab).

import sys, os, subprocess, importlib.util, platform, time

print(f"Python: {sys.version.split()[0]} | {platform.system()} {platform.machine()}")
assert sys.version_info >= (3, 10), (
    "Requer Python ≥ 3.10. No Colab default (3.11+) funciona; "
    "em outra máquina suba para 3.10/3.11/3.12."
)

def _sh(cmd: str) -> int:
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-800:])
    return r.returncode

def _has(pkg: str) -> bool:
    return importlib.util.find_spec(pkg) is not None

_RESTART_FLAG = "/tmp/.atari_drl_restart_done"
already_restarted = os.path.exists(_RESTART_FLAG)
installed_new = False

# 1) gymnasium + ale-py (ROMs embutidas) + stable-baselines3
if not (_has("ale_py") and _has("gymnasium") and _has("stable_baselines3")):
    print("► instalando gymnasium + ale-py + stable-baselines3...")
    _sh('pip install --quiet "gymnasium>=1.0.0" "ale-py>=0.10.0" '
        '"stable-baselines3>=2.4.0" "torch>=2.4"')
    installed_new = True

# 2) Análise + visualização
if not (_has("imageio") and _has("seaborn") and _has("cv2") and _has("scipy")):
    print("► instalando análise + viz...")
    _sh('pip install --quiet "opencv-python-headless" "imageio[ffmpeg]" '
        'tensorboard pandas scipy matplotlib seaborn tqdm rich psutil')
    installed_new = True

# 3) Restart obrigatório se algo foi instalado
if installed_new and not already_restarted:
    open(_RESTART_FLAG, "w").write("1")
    print()
    print("=" * 64)
    print("⚠  PACOTES NOVOS INSTALADOS — REINICIANDO O KERNEL")
    print("   (necessário para numpy/torch recarregarem coerentes)")
    print()
    print("   Após o restart automático: RE-EXECUTE ESTA CÉLULA.")
    print("   Na 2ª execução pula tudo e segue direto para validação.")
    print("=" * 64)
    time.sleep(3)
    os.kill(os.getpid(), 9)   # força restart no Colab/Jupyter

# 4) Validação (só executa se nada foi instalado nesta passagem)
print()
import torch, gymnasium as gym
import ale_py
import stable_baselines3 as sb3
print(f"✓ torch {torch.__version__} | CUDA: {torch.cuda.is_available()}"
      + (f" | GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else ""))
print(f"✓ gymnasium {gym.__version__} | ale-py {ale_py.__version__} | sb3 {sb3.__version__}")

# 5) Smoke test do env: ALE/Pong-v5
gym.register_envs(ale_py)    # registra ALE/<game>-v5 no namespace gymnasium
_env = gym.make("ALE/Pong-v5", render_mode="rgb_array")
_obs, _info = _env.reset(seed=0)
_obs, _r, _term, _trunc, _info = _env.step(_env.action_space.sample())
print(f"✓ env smoke test: obs={_obs.shape}, "
      f"action_space={_env.action_space}, "
      f"info_keys={sorted(_info)[:6]}")
_env.close()

# Limpa flag (próxima execução completa também é válida)
try: os.remove(_RESTART_FLAG)
except OSError: pass

print("\n✓✓ Ambiente pronto.")


Python: 3.12.13 | Linux x86_64
► instalando gymnasium + ale-py + stable-baselines3...
$ pip install --quiet "gymnasium>=1.0.0" "ale-py>=0.10.0" "stable-baselines3>=2.4.0" "torch>=2.4"

⚠  PACOTES NOVOS INSTALADOS — REINICIANDO O KERNEL
   (necessário para numpy/torch recarregarem coerentes)

   Após o restart automático: RE-EXECUTE ESTA CÉLULA.
   Na 2ª execução pula tudo e segue direto para validação.


## 2. Imports

In [ ]:
# --- 2. Imports ---
# stdlib
import os, sys, csv, json, time, shutil, random
from pathlib import Path
from datetime import datetime, timezone

# scientific
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio.v2 as imageio

# RL stack (gymnasium + ALE oficial)
import gymnasium as gym
import ale_py
import torch

# Stable-Baselines3
from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import (
    DummyVecEnv, SubprocVecEnv, VecFrameStack, VecTransposeImage
)

# Registra ALE/<game>-v5 no gymnasium global registry
gym.register_envs(ale_py)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Imports OK | device={DEVICE}")


✓ Imports OK | device=cuda


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 3. Storage (Colab Drive ou local)

In [ ]:
# --- 3. Storage (Colab Drive ou pasta local) ---
# Em Colab: monta o Drive para persistir entre sessões (Colab cai em ~12h).
# Fora do Colab: pasta local ./atari_drl_results.

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive", force_remount=False)
        BASE_DIR = Path("/content/drive/MyDrive/atari_drl_results")
        print(f"✓ Colab + Drive montado em {BASE_DIR}")
    except Exception as e:
        print(f"✗ falha ao montar Drive: {e}")
        BASE_DIR = Path("/content/atari_drl_results")
        print(f"  → caindo para pasta local efêmera: {BASE_DIR}")
else:
    BASE_DIR = Path("./atari_drl_results").resolve()
    print(f"✓ Local: {BASE_DIR}")

# Estrutura:
#  BASE_DIR/
#   ├── models/{algo}/{algo}_{game_short}_{seed}.zip
#   ├── models/checkpoints/{algo}/{tag}/{tag}_{N}_steps.zip
#   ├── logs/{algo}/{tag}_eval.csv
#   ├── logs/tb/{algo}/{tag}/         (TensorBoard)
#   ├── figures/                       (PNG, PDF — gerados pela análise)
#   └── videos/                        (GIFs — verificação visual)
MODELS_DIR = BASE_DIR / "models"
LOGS_DIR   = BASE_DIR / "logs"
FIGS_DIR   = BASE_DIR / "figures"
VIDS_DIR   = BASE_DIR / "videos"
for d in [MODELS_DIR, LOGS_DIR, FIGS_DIR, VIDS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print(f"✓ Pastas prontas em {BASE_DIR}")


Mounted at /content/drive
✓ Colab + Drive montado em /content/drive/MyDrive/atari_drl_results
✓ Pastas prontas em /content/drive/MyDrive/atari_drl_results


## 4. Configuração global

In [ ]:
# --- 4. Configuração global ---
# SMOKE_TEST=True valida o pipeline em ~3-5 min antes de disparar 18h de treino.
# Sempre rode com SMOKE_TEST=True na primeira execução.

SMOKE_TEST = False   # ⚠️ True = 10k timesteps, 1 jogo, 1 seed (sanity check)

# Protocolo experimental
# (env_id, short_name, dificuldade canônica 1-4)
GAMES = [
    ("ALE/Pong-v5",            "Pong",     1),  # mais fácil
    ("ALE/Breakout-v5",        "Breakout", 2),
    ("ALE/MsPacman-v5",        "MsPacman", 3),
    ("ALE/MontezumaRevenge-v5","Montezuma",4),  # mais difícil (sparse reward)
]
GAME_IDS    = [g[0] for g in GAMES]
GAME_SHORTS = [g[1] for g in GAMES]
RANK_CANON  = {g[1]: g[2] for g in GAMES}

# Limiar de "sucesso" por jogo (para a métrica τ adaptada).
# Baseado em: literatura de RL benchmarks + score humano médio (Mnih 2015).
SUCCESS_THRESHOLD = {
    "Pong":      0.0,   # vencer um jogo: score positivo (humano = +9.3)
    "Breakout":  30.0,  # passar de algumas vidas (humano = 31.8)
    "MsPacman":  1000.0,# pontuação razoável (humano = 15693)
    "Montezuma": 400.0, # apenas pegar a 1ª chave (humano = 4753)
}

# Human baseline (Mnih 2015 Nature, Table S2 / Wang et al. 2016) — usado para
# normalizar score na métrica d̄ (análoga à distância normalizada do Mario).
HUMAN_BASELINE = {
    "Pong":     9.3,
    "Breakout": 31.8,
    "MsPacman": 15693.0,
    "Montezuma":4753.0,
}
# Random baseline (idem Mnih 2015) para evitar normalizar contra zero em jogos
# de score negativo (Pong).
RANDOM_BASELINE = {
    "Pong":     -20.7,
    "Breakout":  1.7,
    "MsPacman":  307.3,
    "Montezuma": 0.0,
}

SEEDS           = [42, 123, 2024]
ALGOS           = ["DQN", "PPO", "A2C"]
TOTAL_TIMESTEPS = 500_000
EVAL_FREQ       = 10_000        # avaliação a cada N timesteps de treino
N_EVAL_EPISODES = 5             # 5 eps determinísticos por checkpoint
N_CHECKPOINTS   = 5             # 5 checkpoints uniformes por treino

if SMOKE_TEST:
    print("⚠️  SMOKE_TEST ativo — reduzindo escopo.")
    GAMES_TO_RUN   = [GAMES[0]]     # só Pong
    SEEDS_TO_RUN   = [42]
    TOTAL_TIMESTEPS = 10_000
    EVAL_FREQ       = 2_500
    N_EVAL_EPISODES = 2
    N_CHECKPOINTS   = 2
else:
    GAMES_TO_RUN   = GAMES
    SEEDS_TO_RUN   = SEEDS

print(f"Configuração: {len(GAMES_TO_RUN)} jogos × {len(SEEDS_TO_RUN)} seeds × "
      f"{TOTAL_TIMESTEPS:,} timesteps | eval a cada {EVAL_FREQ:,}")
for g in GAMES_TO_RUN:
    print(f"  • {g[1]:<10s} (dif {g[2]}/4) — {g[0]}")


Configuração: 4 jogos × 3 seeds × 500,000 timesteps | eval a cada 10,000
  • Pong       (dif 1/4) — ALE/Pong-v5
  • Breakout   (dif 2/4) — ALE/Breakout-v5
  • MsPacman   (dif 3/4) — ALE/MsPacman-v5
  • Montezuma  (dif 4/4) — ALE/MontezumaRevenge-v5


## 5. Hiperparâmetros (DQN)

In [ ]:
# --- 5. Hiperparâmetros (DQN) ---
# Valores baseados em SB3 RL Zoo Atari defaults + Mnih 2015 / Schulman 2017.

HPARAMS = {
    "DQN": dict(
        learning_rate            = 1e-4,
        buffer_size              = 100_000,
        learning_starts          = 10_000,
        batch_size               = 32,
        tau                      = 1.0,
        gamma                    = 0.99,
        train_freq               = 4,
        gradient_steps           = 1,
        target_update_interval   = 10_000,
        exploration_fraction     = 0.10,
        exploration_initial_eps  = 1.0,
        exploration_final_eps    = 0.01,
        max_grad_norm            = 10.0,
        # optimize_memory_usage=True: comprime o replay buffer pela metade
        # (não armazena `next_obs` duplicado). Crítico no Colab — com
        # buffer=100k e obs (4,84,84) uint8 o buffer cheio ocuparia ~2.8GB.
        optimize_memory_usage    = True,
        replay_buffer_kwargs     = {"handle_timeout_termination": False},
        n_envs                   = 1,    # DQN off-policy: 1 env é padrão
    ),
}
ALGO_NAME = "DQN"

print(f"✓ Hiperparâmetros carregados: {ALGO_NAME}")
for k, v in HPARAMS[ALGO_NAME].items():
    print(f"  {k:<24s} = {v}")


✓ Hiperparâmetros carregados: DQN
  learning_rate            = 0.0001
  buffer_size              = 100000
  learning_starts          = 10000
  batch_size               = 32
  tau                      = 1.0
  gamma                    = 0.99
  train_freq               = 4
  gradient_steps           = 1
  target_update_interval   = 10000
  exploration_fraction     = 0.1
  exploration_initial_eps  = 1.0
  exploration_final_eps    = 0.01
  max_grad_norm            = 10.0
  optimize_memory_usage    = True
  replay_buffer_kwargs     = {'handle_timeout_termination': False}
  n_envs                   = 1


## 6. Environment factory

In [ ]:
# --- 6. Environment factory ---
# Pipeline de pré-processamento Atari canônico (Mnih 2015):
#   gym.make(ALE/<game>-v5) → AtariWrapper → Monitor → VecEnv → FrameStack(4) → Transpose
#
# AtariWrapper da SB3 já faz: NoopReset, MaxAndSkip(4), EpisodicLife,
# FireReset, WarpFrame(84,84), ClipReward. É o "Atari preset" padrão da SB3.
#
# Observação final: (4, 84, 84) uint8 — input do CnnPolicy da SB3.

def set_global_seed(seed: int) -> None:
    """Fixa todas as fontes de aleatoriedade."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_atari_env(env_id: str, seed: int = 0,
                   render_mode: str = "rgb_array",
                   clip_reward: bool = True,
                   terminal_on_life_loss: bool = True):
    """Factory para UM env Atari (não vetorizado).

    Retorna a thunk (closure) para uso com SubprocVecEnv/DummyVecEnv —
    a construção real acontece dentro do worker process.

    Args:
        clip_reward: se True (treino), recompensa truncada em sign({-1,0,+1}).
                     Para AVALIAÇÃO, passar False para preservar o score real.
        terminal_on_life_loss: se True (treino), perder uma vida termina o
                     episódio. Para AVALIAÇÃO passar False para contar
                     score acumulado ao longo de TODAS as vidas (3-5).
    """
    def _thunk():
        # Imports DENTRO do thunk: defensivo contra spawn (vs fork) em
        # SubprocVecEnv. Em alguns ambientes (Windows, ou Python 3.13+)
        # o subprocess pode não herdar imports do notebook.
        import gymnasium as _gym
        import ale_py as _ale
        from stable_baselines3.common.atari_wrappers import AtariWrapper as _AW
        from stable_baselines3.common.monitor import Monitor as _Mon

        _gym.register_envs(_ale)
        env = _gym.make(env_id, render_mode=render_mode)
        # AtariWrapper aceita estes flags diretamente:
        env = _AW(env,
                  noop_max=30,
                  frame_skip=4,
                  screen_size=84,
                  terminal_on_life_loss=terminal_on_life_loss,
                  clip_reward=clip_reward)
        env = _Mon(env)
        return env
    return _thunk


def make_vec_env_atari(env_id: str, n_envs: int = 1, seed: int = 0,
                        use_subproc: bool | None = None,
                        clip_reward: bool = True,
                        terminal_on_life_loss: bool = True):
    """VecEnv com n cópias + FrameStack(4) + Transpose para CHW.

    n_envs=1  → DummyVecEnv (sem overhead de subprocess)
    n_envs>1  → SubprocVecEnv (paralelismo real via spawn/fork)
    """
    if use_subproc is None:
        use_subproc = n_envs > 1
    thunks = [make_atari_env(env_id, seed=seed + i,
                              clip_reward=clip_reward,
                              terminal_on_life_loss=terminal_on_life_loss)
              for i in range(n_envs)]
    VecCls = SubprocVecEnv if use_subproc else DummyVecEnv
    venv = VecCls(thunks)
    venv = VecFrameStack(venv, n_stack=4, channels_order="last")
    venv = VecTransposeImage(venv)
    venv.seed(seed)
    return venv


# Sanity check: dimensão da obs
_smoke = make_vec_env_atari(GAMES_TO_RUN[0][0], n_envs=1, seed=0)
_obs = _smoke.reset()
print(f"✓ Env factory OK | obs shape: {_obs.shape} (esperado: (1, 4, 84, 84))")
assert _obs.shape == (1, 4, 84, 84), f"Pipeline quebrado: obs={_obs.shape}"
_smoke.close()


✓ Env factory OK | obs shape: (1, 4, 84, 84) (esperado: (1, 4, 84, 84))


## 7. Callback de avaliação

In [ ]:
# --- 7. Callback de avaliação (Grupo I + Grupo II) ---
# A cada EVAL_FREQ timesteps de treino, roda N_EVAL_EPISODES episódios
# determinísticos no eval_env (seed descorrelacionada do treino) e escreve
# UMA LINHA POR EPISÓDIO no CSV — persistência incremental sobrevive a crash.
#
# IMPORTANTE: eval_env usa clip_reward=False e terminal_on_life_loss=False
# para coletar o SCORE REAL do jogo (não o sinal truncado de treino) e
# permitir contagem real de mortes ao longo de TODAS as vidas (3-5/episódio).

class AtariEvalCallback(BaseCallback):
    """Avalia determinístico e persiste métricas Grupo I + Grupo II em CSV."""

    HEADER = [
        "algo", "game", "seed",
        "timestep", "ep_idx",
        "ep_reward",         # score REAL (sem clip)
        "ep_length",         # passos da policy
        "deaths",            # vidas perdidas (info['lives'])
        "ts_iso",
    ]

    def __init__(self, eval_env, *, eval_freq: int, n_eval_episodes: int,
                 log_path, algo: str, game: str, seed: int, verbose: int = 0):
        super().__init__(verbose)
        self.eval_env        = eval_env
        self.eval_freq       = max(int(eval_freq), 1)
        self.n_eval_episodes = int(n_eval_episodes)
        self.log_path        = Path(log_path)
        self.algo, self.game, self.seed = algo, game, seed
        self._last_eval      = 0
        self._init_log()

    def _init_log(self) -> None:
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.log_path.exists():
            with open(self.log_path, "w", newline="") as f:
                csv.writer(f).writerow(self.HEADER)

    def _on_step(self) -> bool:
        # num_timesteps cresce em n_envs por step do worker — comparar contra
        # o valor absoluto garante o intervalo solicitado.
        if self.num_timesteps - self._last_eval >= self.eval_freq:
            self._last_eval = self.num_timesteps
            self._evaluate()
        return True

    def _evaluate(self) -> None:
        rows = []
        for ep_idx in range(self.n_eval_episodes):
            obs = self.eval_env.reset()
            done = np.array([False])
            ep_r, ep_len, deaths = 0.0, 0, 0
            prev_lives = None
            while not done.any():
                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, done, infos = self.eval_env.step(action)
                ep_r  += float(reward[0])
                ep_len += 1
                info = infos[0] if isinstance(infos, (list, tuple)) else infos
                # info['lives'] vem do AtariWrapper / ALE
                cur_lives = info.get("lives", None)
                if (prev_lives is not None and cur_lives is not None
                        and cur_lives < prev_lives):
                    deaths += (prev_lives - cur_lives)
                prev_lives = cur_lives
            rows.append([
                self.algo, self.game, self.seed,
                int(self.num_timesteps), ep_idx,
                round(ep_r, 4), int(ep_len), int(deaths),
                datetime.now(timezone.utc).isoformat(timespec="seconds"),
            ])

        with open(self.log_path, "a", newline="") as f:
            csv.writer(f).writerows(rows)

        med_r = float(np.median([r[5] for r in rows]))
        med_d = float(np.median([r[7] for r in rows]))
        print(f"  [{self.algo}/{self.game}/seed={self.seed}] "
              f"ts={self.num_timesteps:>7,} | med_score={med_r:7.1f} | "
              f"med_deaths={med_d:.1f}")

print("✓ AtariEvalCallback definido")


✓ AtariEvalCallback definido


## 8. Função de treinamento

In [ ]:
# --- 8. Função de treinamento ---
# train_one(algo, game_id, seed):
#   - idempotente: pula se o modelo final existe
#   - resume from checkpoint: detecta último checkpoint salvo e continua
#     (essencial p/ Colab — sessões caem em ~12h)
#   - persiste 5 checkpoints uniformes em MODELS_DIR/checkpoints/{algo}/{tag}/
#   - escreve CSV de avaliação incrementalmente (sobrevive a crash)

_ALGO_CLS = {"DQN": DQN, "PPO": PPO, "A2C": A2C}

# game_id "ALE/Pong-v5" → tag-safe "Pong"
def _game_short(game_id: str) -> str:
    short = game_id.split("/")[-1].replace("-v5", "")
    # Mapeia "MontezumaRevenge" → "Montezuma" para consistência com GAME_SHORTS
    if short.startswith("MontezumaRevenge"):
        return "Montezuma"
    return short


def _find_latest_ckpt(ckpt_dir, tag: str):
    """Retorna (path, completed_steps) do checkpoint mais recente, ou (None, 0)."""
    ckpt_dir = Path(ckpt_dir)
    if not ckpt_dir.exists():
        return None, 0
    ckpts = list(ckpt_dir.glob(f"{tag}_*_steps.zip"))
    if not ckpts:
        return None, 0
    def _n(p):
        try:
            return int(p.stem.split("_")[-2])
        except (ValueError, IndexError):
            return -1
    ckpts = [p for p in ckpts if _n(p) > 0]
    if not ckpts:
        return None, 0
    latest = max(ckpts, key=_n)
    return latest, _n(latest)


def train_one(algo: str, game_id: str, seed: int,
              total_timesteps: int = None,
              eval_freq: int = None,
              n_eval_episodes: int = None) -> dict:
    """Treina UMA configuração (algo, game, seed) com idempotência + resume."""
    total_timesteps  = total_timesteps  or TOTAL_TIMESTEPS
    eval_freq        = eval_freq        or EVAL_FREQ
    n_eval_episodes  = n_eval_episodes  or N_EVAL_EPISODES

    game = _game_short(game_id)
    tag  = f"{algo}_{game}_{seed}"
    final_path = MODELS_DIR / algo / f"{tag}.zip"
    ckpt_dir   = MODELS_DIR / "checkpoints" / algo / tag
    eval_log   = LOGS_DIR / algo / f"{tag}_eval.csv"
    final_path.parent.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    if final_path.exists():
        print(f"✓ [{tag}] modelo final já existe — pulando.")
        return {"status": "skipped", "tag": tag, "path": str(final_path)}

    set_global_seed(seed)
    n_envs   = HPARAMS[algo].get("n_envs", 1)
    hp       = {k: v for k, v in HPARAMS[algo].items() if k != "n_envs"}
    ModelCls = _ALGO_CLS[algo]

    # TREINO: clip_reward=True, terminal_on_life_loss=True (config Mnih 2015)
    train_env = make_vec_env_atari(game_id, n_envs=n_envs, seed=seed,
                                    clip_reward=True,
                                    terminal_on_life_loss=True)
    # AVALIAÇÃO: clip_reward=False (score real), terminal_on_life_loss=False
    # (deixa o episódio rodar até game over para contar mortes total e score)
    eval_env  = make_vec_env_atari(game_id, n_envs=1, seed=seed + 9999,
                                    clip_reward=False,
                                    terminal_on_life_loss=False)

    try:
        # --- Resume ou início ---
        latest_ckpt, completed = _find_latest_ckpt(ckpt_dir, tag)
        if latest_ckpt is not None:
            remaining = total_timesteps - completed
            if remaining <= 0:
                print(f"✓ [{tag}] checkpoint cobre o total — promovendo a final.")
                shutil.copy(latest_ckpt, final_path)
                return {"status": "promoted", "tag": tag, "path": str(final_path)}
            print(f"↻ [{tag}] retomando de {completed:,} ts → faltam {remaining:,}")
            model = ModelCls.load(latest_ckpt, env=train_env, device=DEVICE)
            reset_steps = False
        else:
            remaining   = total_timesteps
            reset_steps = True
            print(f"► [{tag}] iniciando do zero ({total_timesteps:,} timesteps)")
            model = ModelCls(
                "CnnPolicy", train_env,
                seed=seed, verbose=0, device=DEVICE,
                tensorboard_log=str(LOGS_DIR / "tb" / algo),
                **hp,
            )

        # --- Callbacks ---
        # save_freq é por step do worker principal: em vec_env multi-worker,
        # total_timesteps reais por iteração = n_envs. Para 5 checkpoints
        # uniformes sobre TODO o treino:
        save_freq = max((total_timesteps // N_CHECKPOINTS) // n_envs, 1)
        cb_ckpt = CheckpointCallback(
            save_freq=save_freq, save_path=str(ckpt_dir), name_prefix=tag,
            save_replay_buffer=False, save_vecnormalize=False,
        )
        cb_eval = AtariEvalCallback(
            eval_env, eval_freq=max(eval_freq // n_envs, 1),
            n_eval_episodes=n_eval_episodes, log_path=eval_log,
            algo=algo, game=game, seed=seed,
        )

        # --- Treino ---
        t0 = time.time()
        model.learn(
            total_timesteps    = remaining,
            callback           = [cb_ckpt, cb_eval],
            reset_num_timesteps= reset_steps,
            tb_log_name        = tag,
            progress_bar       = True,
        )
        elapsed = time.time() - t0

        model.save(final_path)
        print(f"✓ [{tag}] concluído em {elapsed/60:.1f} min → {final_path.name}")
        return {"status": "completed", "tag": tag, "path": str(final_path),
                "elapsed_sec": elapsed}

    finally:
        try: train_env.close()
        except Exception: pass
        try: eval_env.close()
        except Exception: pass

print("✓ train_one() definido (idempotente + resume from checkpoint)")


✓ train_one() definido (idempotente + resume from checkpoint)


## 9. Loop de treinamento (DQN)

In [ ]:
# --- 9. Loop de treinamento (DQN) ---
# Roda TODAS as combinações (game, seed) para o algoritmo deste notebook.
# Cada run é independente e idempotente — se a sessão Colab cair, basta
# re-executar esta célula que ela retoma do último checkpoint.
#
# Tempo estimado em T4 (Colab Pro):
#   DQN  (n_envs=1):  ~80 min × 12 runs = ~16h
#   PPO  (n_envs=8):  ~30 min × 12 runs = ~6h
#   A2C  (n_envs=16): ~20 min × 12 runs = ~4h

def run_dqn_experiments():
    """Executa as 12 combinações de DQN ({4 jogos} × {3 seeds})."""
    results, total = [], len(GAMES_TO_RUN) * len(SEEDS_TO_RUN)
    i = 0
    for game in GAMES_TO_RUN:
        game_id, game_short, _diff = game
        for seed in SEEDS_TO_RUN:
            i += 1
            print(f"\n{'='*70}")
            print(f"[{i}/{total}] DQN | {game_short} | seed {seed}")
            print('='*70)
            try:
                res = train_one("DQN", game_id, seed)
            except Exception as e:
                res = {"status": "failed",
                       "tag": f"DQN_{game_short}_{seed}",
                       "error": str(e)}
                print(f"✗ FALHA [{res['tag']}]: {e}")
            results.append(res)
    print(f"\n{'='*70}\nResumo DQN:")
    for r in results:
        print(f"  {r['status']:>10s} | {r['tag']}")
    return results

# ▶︎ Descomente a linha abaixo para disparar o treino completo.
#   (Mantenha SMOKE_TEST=True na primeira execução p/ validar o pipeline.)
results = run_dqn_experiments()


 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 499,999/500,000  [ 2:52:12 < 0:00:01 , 104 it/s ]

[DQN/Montezuma/seed=2024] ts=500,000 | med_score=    0.0 | med_deaths=0.0

✓ [DQN_Montezuma_2024] concluído em 172.2 min → DQN_Montezuma_2024.zip

Resumo DQN:
     skipped | DQN_Pong_42
   completed | DQN_Pong_123
   completed | DQN_Pong_2024
   completed | DQN_Breakout_42
   completed | DQN_Breakout_123
   completed | DQN_Breakout_2024
   completed | DQN_MsPacman_42
   completed | DQN_MsPacman_123
   completed | DQN_MsPacman_2024
   completed | DQN_Montezuma_42
   completed | DQN_Montezuma_123
   completed | DQN_Montezuma_2024


## 10. Verificação visual

In [ ]:
# --- 10. Verificação visual ---
# Renderiza o agente treinado jogando um episódio e salva GIF.
#
# CRÍTICO: env.render() em emuladores retorna referência ao buffer interno.
# Sem .copy() em cada frame, todas as entradas da lista apontam para o
# mesmo bloco de memória → GIF estático.

def render_agent_episode(model, game_id: str, *,
                         max_steps: int = 3000, fps: int = 30,
                         seed: int = 999, save_path=None) -> list:
    """Roda 1 episódio determinístico, captura frames RGB e (opcional) salva GIF.

    Retorna a lista de frames para uso em FuncAnimation.to_jshtml().
    """
    # Guardamos referência ao env "raw" (antes do WarpFrame que reduz para 84x84)
    # para chamar render() e obter o frame RGB (210x160) completo.
    raw_holder = {}
    def _thunk():
        import gymnasium as _gym
        import ale_py as _ale
        from stable_baselines3.common.atari_wrappers import AtariWrapper as _AW
        from stable_baselines3.common.monitor import Monitor as _Mon
        _gym.register_envs(_ale)
        env = _gym.make(game_id, render_mode="rgb_array")
        raw_holder["env"] = env   # raw (sem AtariWrapper)
        env = _AW(env, noop_max=30, frame_skip=4, screen_size=84,
                  terminal_on_life_loss=False, clip_reward=False)
        env = _Mon(env)
        return env

    venv = DummyVecEnv([_thunk])
    venv = VecFrameStack(venv, n_stack=4, channels_order="last")
    venv = VecTransposeImage(venv)
    venv.seed(seed)
    obs = venv.reset()
    raw = raw_holder["env"]

    frames, total_r, n_steps = [], 0.0, 0
    for _ in range(max_steps):
        frame = raw.render()
        if frame is not None:
            frames.append(frame.copy())   # ← evita aliasing do buffer
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, _infos = venv.step(action)
        total_r += float(reward[0])
        n_steps += 1
        if done[0]:
            f = raw.render()
            if f is not None:
                frames.append(f.copy())
            break

    venv.close()
    print(f"✓ episódio: {n_steps} passos | score={total_r:.1f} | {len(frames)} frames")

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        # duration em MILISSEGUNDOS
        imageio.mimsave(save_path, frames,
                        duration=int(round(1000 / fps)), loop=0)
        print(f"✓ GIF salvo: {save_path}")

    return frames


def show_frames_inline(frames, fps: int = 30, title: str = ""):
    """Anima frames inline via FuncAnimation.to_jshtml() — funciona em
    VSCode, JupyterLab, Colab (Image(filename=GIF) falha em alguns)."""
    from matplotlib import animation
    from IPython.display import HTML
    if not frames:
        print("(sem frames)"); return
    fig, ax = plt.subplots(figsize=(5, 4.5))
    im = ax.imshow(frames[0]); ax.axis("off")
    if title: ax.set_title(title)
    def _upd(i):
        im.set_data(frames[i])
        return [im]
    anim = animation.FuncAnimation(fig, _upd, frames=len(frames),
                                   interval=1000/fps, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


# Exemplo (descomente após treinar):
# tag = "{ALGO_NAME}_Pong_42"
# model = _ALGO_CLS["{ALGO_NAME}"].load(MODELS_DIR / "{ALGO_NAME}" / f"{tag}.zip")
# frames = render_agent_episode(model, "ALE/Pong-v5",
#                                 save_path=VIDS_DIR / f"{tag}.gif")
# show_frames_inline(frames, title=tag)


In [ ]:
# Exemplo (descomente após treinar):
tag = f"{ALGO_NAME}_Pong_42"
model = _ALGO_CLS[f"{ALGO_NAME}"].load(MODELS_DIR / f"{ALGO_NAME}" / f"{tag}")
frames = render_agent_episode(model, "ALE/Pong-v5",
                                save_path=VIDS_DIR / f"{tag}.gif")
show_frames_inline(frames, title=tag)

✓ episódio: 187 passos | score=-21.0 | 188 frames
✓ GIF salvo: /content/drive/MyDrive/atari_drl_results/videos/DQN_Pong_42.gif


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Próximos passos

1. **Smoke test** — mantenha `SMOKE_TEST = True` na célula 9 e execute o
   notebook inteiro. Tempo: ~3-5 min. Confirma que o pipeline está OK.

2. **Treino completo** — mude para `SMOKE_TEST = False`, descomente a linha
   `results = run_dqn_experiments()` na célula 19, e execute.

3. **Se a sessão Colab cair** — basta reabrir o notebook e re-executar
   as células 1-19. Drive remonta, instalações são puladas, modelos finais
   existentes são pulados, e os runs incompletos retomam do último checkpoint.

4. **Verificação visual** — célula 21 carrega o modelo de uma config e gera um
   GIF do agente jogando. Use para conferir qualitativamente o aprendizado.

5. **Outros algoritmos** — abra 02_train_ppo.ipynb e 03_train_a2c.ipynb e repita.

6. **Análise final** — quando os 36 runs estiverem completos, abra
   `04_analysis.ipynb` para computar métricas, testes estatísticos e
   gerar todas as figuras do paper.